# Phionyx Core — Hello in 5 Minutes

A zero-to-running tour of the SDK. No LLM, no server, no API key. Every
cell runs on a fresh `pip install phionyx-core`.

```bash
pip install phionyx-core
```

What this notebook covers:

1. The state vector
2. Φ — cognitive resonance
3. Determinism check
4. Kill switch
5. The 46-block canonical pipeline
6. Where to go next

Total run time on a recent laptop: under 30 seconds.


## 0. Imports + version

In [1]:
import phionyx_core
from phionyx_core import (
    EchoState2,
    calculate_phi_v2_1,
    calculate_phi_cognitive,
)
from phionyx_core.governance.kill_switch import KillSwitch
from phionyx_core.contracts.telemetry import get_canonical_blocks

print("phionyx_core:", phionyx_core.__version__)

phionyx_core: 0.2.1


## 1. The state vector

Phionyx represents cognitive state as a small structured vector with three
primary attributes — arousal `A`, valence `V`, entropy `H` — plus derived
quantities like `resonance` and `stability` that are computed on demand
and never stored. That keeps the canonical state minimal and reproducible.


In [2]:
state = EchoState2(A=0.6, V=0.3, H=0.4)

print(f"Primary    A={state.A}  V={state.V}  H={state.H}")
print(f"Derived    resonance={state.resonance:.4f}  stability={state.stability:.4f}")

Primary    A=0.6  V=0.3  H=0.4
Derived    resonance=0.4800  stability=0.6000


## 2. Φ — Cognitive resonance

`calculate_phi_v2_1` is the canonical Hybrid Resonance Model. Same
inputs, same outputs, byte for byte — see the next cell.


In [3]:
phi = calculate_phi_v2_1(
    valence=state.V,
    arousal=state.A,
    amplitude=state.A * 10.0,   # arousal-to-amplitude scaling
    time_delta=0.1,
    gamma=0.15,
    stability=state.stability,
    entropy=state.H,
    w_c=0.6,                    # cognitive weight
    w_p=0.4,                    # physical weight
)

for k, v in phi.items():
    print(f"  {k:<18} {v:.4f}")

  phi                1.5250
  phi_cognitive      0.1774
  phi_physical       3.5464
  weight_cognitive   0.6000
  weight_physical    0.4000


## 3. Determinism check

If anything in the substrate were stochastic, the next cell would print a
number greater than 1. It will print exactly `1`.


In [4]:
inputs = dict(
    valence=0.5, arousal=0.7, amplitude=5.0, time_delta=1.0,
    gamma=0.15, stability=0.9, entropy=0.3, w_c=0.75, w_p=0.25,
)
unique = {round(calculate_phi_v2_1(**inputs)["phi"], 12) for _ in range(100)}
print(f"unique Φ values across 100 runs: {len(unique)}")
print(f"Φ = {next(iter(unique))}")

unique Φ values across 100 runs: 1
Φ = 1.065914479372


## 4. Kill switch

`KillSwitch` is the runtime safety primitive. It is **fail-closed** — any
evaluation error or out-of-band signal puts the agent into a locked
state that no prompt can talk past.


In [5]:
ks = KillSwitch()
print(f"initial state: {ks.state.value}  (armed = ready to fire if needed)")

# A healthy turn: low ethics risk, high meta-trust, no drift.
ok = ks.evaluate(ethics_max_risk=0.10, t_meta=0.85, drift_detected=False, turn_id=1)
print(f"\nhealthy turn → triggered={ok.triggered}")

# A catastrophic ethics signal trips the gate.
trip = ks.evaluate(ethics_max_risk=0.97, t_meta=0.8, turn_id=2)
print(f"ethics 0.97   → triggered={trip.triggered}  trigger={trip.trigger.value}")

KILL SWITCH TRIGGERED: ethics_max_risk_exceeded — Ethics max risk 0.970 exceeds threshold 0.95


initial state: armed  (armed = ready to fire if needed)

healthy turn → triggered=False
ethics 0.97   → triggered=True  trigger=ethics_max_risk_exceeded


## 5. The 46-block canonical pipeline

The pipeline contract is data, not code — versioned and inspectable. The
current contract is v3.8.0, 46 blocks. Each block declares a determinism
class (`strict`, `seeded`, or `noisy_sensor`).


In [6]:
blocks = get_canonical_blocks()
print(f"canonical blocks (v3.8.0): {len(blocks)}\n")

print("first five (perception side):")
for i, b in enumerate(blocks[:5], start=1):
    print(f"  {i:>2}. {b}")

print("\nlast five (reflect / commit side):")
for i, b in enumerate(blocks[-5:], start=len(blocks) - 4):
    print(f"  {i:>2}. {b}")

canonical blocks (v3.8.0): 46

first five (perception side):
   1. kill_switch_gate
   2. time_update_sot
   3. input_safety_gate
   4. intent_classification
   5. context_retrieval_rag

last five (reflect / commit side):
  42. response_build
  43. memory_consolidation
  44. audit_layer
  45. outcome_feedback
  46. learning_gate


## What's next

You've seen the substrate. The other notebooks go deeper:

- [`01_determinism_and_physics.ipynb`](01_determinism_and_physics.ipynb) — 1000-run determinism proof, valence × arousal Φ heatmap, side-by-side with a noisy alternative.
- [`02_kill_switch_in_action.ipynb`](02_kill_switch_in_action.ipynb) — all four kill-switch triggers, NaN fail-closed guard, tamper-evident event log.
- [`03_pipeline_blocks_and_audit.ipynb`](03_pipeline_blocks_and_audit.ipynb) — implement and run a custom `PipelineBlock`.

For the orchestrator path (governance pipeline + LLM provider), see
[`examples/fastapi/`](../fastapi/).

Architecture deep dive: [Phionyx README](https://github.com/halvrenofviryel/phionyx-research#readme).

Thanks for trying it.
